# 03 — Task 2: target, history features and temporal splits
The interval-end convention is a declared user-accepted assumption. Retrospective archived observations are used; real-time performance is not claimed.

In [ ]:
from pathlib import Path
import os
import sys

from IPython.display import display
from pyspark.sql import SparkSession, functions as F

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path[:0] = [str(PROJECT_ROOT / "src"), str(PROJECT_ROOT)]

from rainfall.config import load_config
from rainfall.runtime import spark_action

BUCKET = os.environ.get("BUCKET", "sg-rainfall-dsa5208-261011-data")
RUN_ID = os.environ.get("RUN_ID", "20261012-full-01")
ROOT = f"gs://{BUCKET}/runs/{RUN_ID}"
config = load_config(PROJECT_ROOT / "config" / "experiment.json")
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")
spark.conf.set("spark.sql.ansi.enabled", "false")

In [ ]:
from rainfall.fixtures import run_fixtures
from rainfall.features import build_base_features, assign_splits, feature_dictionary
with spark_action(spark, RUN_ID, "target_and_leakage_fixtures"):
    display(run_fixtures(spark, config))
display(feature_dictionary(config))

In [ ]:
with spark_action(spark, RUN_ID, "base_features_and_labels"):
    clean = spark.read.parquet(f"{ROOT}/cleaned")
    base = assign_splits(build_base_features(clean, config), config)
    base.write.mode("errorifexists").partitionBy("split").parquet(f"{ROOT}/base_features")
    attrition = base.groupBy("split", "label_valid").agg(F.count("*").alias("base_samples"), F.min("epoch").alias("first_prediction_epoch"), F.max("epoch").alias("last_prediction_epoch"))
    attrition.write.mode("errorifexists").json(f"{ROOT}/sample_attrition")
    attrition.show(20, truncate=False)
    display({"decision_grid_policy": "observed exact-grid station/time keys; missing current observation times excluded from training/evaluation", "excluded_current_observation_slots": "documented by full-calendar coverage denominators", "history_policy": "partial windows retained with count/coverage indicators and train-fitted median imputation"})

In [ ]:
spark.stop()